In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Demographics"
)

xpt_file = DATA_DIR / "DEMO_H.xpt"
csv_file = DATA_DIR / "DEMO_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2013-2014 DEMO_H XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SDDSRVYR",
    "RIDSTATR",
    "RIAGENDR",
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDRETH1",
    "RIDRETH3",
    "RIDEXMON",
    "RIDEXAGM",
    "DMQMILIZ",
    "DMQADFC",
    "DMDBORN4",
    "DMDCITZN",
    "DMDYRSUS",
    "DMDEDUC3",
    "DMDEDUC2",
    "DMDMARTL",
    "RIDEXPRG",
    "SIALANG",
    "SIAPROXY",
    "SIAINTRP",
    "FIALANG",
    "FIAPROXY",
    "FIAINTRP",
    "MIALANG",
    "MIAPROXY",
    "MIAINTRP",
    "AIALANGA",
    "DMDHHSIZ",
    "DMDFMSIZ",
    "DMDHHSZA",
    "DMDHHSZB",
    "DMDHHSZE",
    "DMDHRGND",
    "DMDHRAGE",
    "DMDHRBR4",
    "DMDHREDU",
    "DMDHRMAR",
    "DMDHSEDU",
    "WTINT2YR",
    "WTMEC2YR",
    "SDMVPSU",
    "SDMVSTRA",
    "INDHHIN2",
    "INDFMIN2",
    "INDFMPIR"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DEMO_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 10175:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 47:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,175 rows x 47 columns."
)


# ============================================================
# 4. DEFINE DECIMAL VS WHOLE-NUMBER VARIABLES
# ============================================================

#
# CRITICAL:
#
# These are genuine continuous decimal fields:
#
# WTINT2YR
#     Full sample 2-year interview weight
#
# WTMEC2YR
#     Full sample 2-year MEC examination weight
#
# INDFMPIR
#     Family income-to-poverty ratio
#
# DO NOT:
#
#     round()
#     convert to Int64
#     use float_format during export
#

DECIMAL_COLUMNS = [
    "WTINT2YR",
    "WTMEC2YR",
    "INDFMPIR"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly three decimal fields."
    )


if len(INTEGER_COLUMNS) != 44:

    raise ValueError(
        "Expected exactly 44 whole-number fields."
    )


print("\n--- VARIABLE TYPE PLAN ---")


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace this value before validating
# the exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print(
    "\nTiny values by variable:"
)


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached DEMO_H.xpt pattern:
#
# RIDAGEYR   =   405
# RIDAGEMN   =    28
# RIDEXAGM   =     4
# DMDEDUC3   =   277
# DMDHHSZA   = 6,417
# DMDHHSZB   = 4,907
# DMDHHSZE   = 7,384
# WTMEC2YR   =   362
# INDFMPIR   =   105
#
# TOTAL      = 19,889
#

EXPECTED_TINY_COUNTS = {
    "RIDAGEYR": 405,
    "RIDAGEMN": 28,
    "RIDEXAGM": 4,
    "DMDEDUC3": 277,
    "DMDHHSZA": 6417,
    "DMDHHSZB": 4907,
    "DMDHHSZE": 7384,
    "WTMEC2YR": 362,
    "INDFMPIR": 105
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DEMO_H pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 19889:

    raise ValueError(
        f"Unexpected total tiny count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact 19,889-value tiny-zero "
    "pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Restore only the nine validated variables.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 19889:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# source should contain.
#
# ONLY validated tiny -> zero values may change.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# MUST run BEFORE Int64 conversion.
#
# Expected genuine decimal variables:
#
# WTINT2YR
# WTMEC2YR
# INDFMPIR
#

print("\n--- GENUINE DECIMAL CHECK ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{fractional_count:,} fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern changed.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 3 genuine decimal fields identified."
)


# ============================================================
# 11. DECIMAL FIELD DETAILS
# ============================================================

print("\n--- DECIMAL FIELD DETAILS ---")


for col in DECIMAL_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    print(
        f"\n{col}:"
    )


    print(
        "  Nonmissing:",
        f"{len(values):,}"
    )


    print(
        "  Fractional:",
        f"{int(fractional_mask.sum()):,}"
    )


    print(
        "  Whole-looking decimals:",
        f"{int((~fractional_mask).sum()):,}"
    )


    print(
        "  Minimum:",
        np.min(values)
    )


    print(
        "  Maximum:",
        np.max(values)
    )


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 73557
    and
    df["SEQN"].max() == 83731
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 13. FORMAT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert only the 44 proven whole-number fields.
#
# EXCLUDE:
#
# WTINT2YR
# WTMEC2YR
# INDFMPIR
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine decimal values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 44 proven whole-number fields "
    "converted to nullable Int64."
)


print(
    "PASS: WTINT2YR, WTMEC2YR and INDFMPIR "
    "remain genuine decimal fields."
)


# ============================================================
# 14. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing=0
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches expected release."
    )


# ============================================================
# 15. CDC CHECK - SDDSRVYR
# ============================================================

check_frequency(
    "SDDSRVYR",
    {
        8: 10175
    }
)


# ============================================================
# 16. CDC CHECK - RIDSTATR
# ============================================================

check_frequency(
    "RIDSTATR",
    {
        1: 362,
        2: 9813
    }
)


# ============================================================
# 17. CDC CHECK - RIAGENDR
# ============================================================

check_frequency(
    "RIAGENDR",
    {
        1: 5003,
        2: 5172
    }
)


# ============================================================
# 18. CDC CHECK - RIDAGEYR
# ============================================================

#
# Age in years.
#
# 0 is valid for infants <1 year.
#

s = df["RIDAGEYR"]


if not (
    int(
        s.between(
            0,
            79
        ).sum()
    ) == 9823

    and int(
        s.eq(0)
        .sum()
    ) == 405

    and int(
        s.eq(80)
        .sum()
    ) == 352

    and int(
        s.isna()
        .sum()
    ) == 0
):

    raise ValueError(
        "RIDAGEYR differs from expected release."
    )


print(
    "PASS: RIDAGEYR matches release; "
    "405 valid age-zero records."
)


# ============================================================
# 19. CDC CHECK - RIDAGEMN
# ============================================================

#
# Age in months for young children.
#

s = df["RIDAGEMN"]


if not (
    int(
        s.between(
            0,
            24
        ).sum()
    ) == 673

    and int(
        s.eq(0)
        .sum()
    ) == 28

    and int(
        s.isna()
        .sum()
    ) == 9502
):

    raise ValueError(
        "RIDAGEMN differs from expected release."
    )


print(
    "PASS: RIDAGEMN matches release; "
    "28 valid zero-month records."
)


# ============================================================
# 20. CDC CHECK - RIDRETH1 / RIDRETH3
# ============================================================

check_frequency(
    "RIDRETH1",
    {
        1: 1730,
        2: 960,
        3: 3674,
        4: 2267,
        5: 1544
    }
)


check_frequency(
    "RIDRETH3",
    {
        1: 1730,
        2: 960,
        3: 3674,
        4: 2267,
        6: 1074,
        7: 470
    }
)


# ============================================================
# 21. CDC CHECK - RIDEXMON
# ============================================================

check_frequency(
    "RIDEXMON",
    {
        1: 4823,
        2: 4990
    },
    362
)


# ============================================================
# 22. CDC CHECK - RIDEXAGM
# ============================================================

s = df["RIDEXAGM"]


if not (
    int(
        s.between(
            0,
            239
        ).sum()
    ) == 4213

    and int(
        s.eq(0)
        .sum()
    ) == 4

    and int(
        s.isna()
        .sum()
    ) == 5962
):

    raise ValueError(
        "RIDEXAGM differs from expected release."
    )


print(
    "PASS: RIDEXAGM matches release; "
    "4 valid zero-month exam-age records."
)


# ============================================================
# 23. CDC CHECK - DMDEDUC3
# ============================================================

#
# Education level age 6-19.
#
# 0 is valid:
#
#     Never attended / kindergarten only
#

EXPECTED_DMDEDUC3 = {
    0: 277,
    1: 237,
    2: 231,
    3: 215,
    4: 234,
    5: 212,
    6: 178,
    7: 203,
    8: 189,
    9: 194,
    10: 162,
    11: 186,
    12: 45,
    13: 137,
    14: 9,
    15: 81,
    55: 3,
    66: 9,
    99: 1
}


check_frequency(
    "DMDEDUC3",
    EXPECTED_DMDEDUC3,
    7372
)


# ============================================================
# 24. HOUSEHOLD/FAMILY SIZE
# ============================================================

check_frequency(
    "DMDHHSIZ",
    {
        1: 817,
        2: 1787,
        3: 1779,
        4: 2100,
        5: 1781,
        6: 985,
        7: 926
    }
)


check_frequency(
    "DMDFMSIZ",
    {
        1: 1297,
        2: 1610,
        3: 1737,
        4: 2027,
        5: 1723,
        6: 961,
        7: 820
    }
)


# ============================================================
# 25. CDC CHECK - DMDHHSZA
# ============================================================

#
# Number of children <=5 in household.
#

check_frequency(
    "DMDHHSZA",
    {
        0: 6417,
        1: 2341,
        2: 1068,
        3: 349
    }
)


# ============================================================
# 26. CDC CHECK - DMDHHSZB
# ============================================================

#
# Number of children age 6-17.
#

check_frequency(
    "DMDHHSZB",
    {
        0: 4907,
        1: 2099,
        2: 1814,
        3: 887,
        4: 468
    }
)


# ============================================================
# 27. CDC CHECK - DMDHHSZE
# ============================================================

#
# Number of adults >=60.
#

check_frequency(
    "DMDHHSZE",
    {
        0: 7384,
        1: 1612,
        2: 1114,
        3: 65
    }
)


# ============================================================
# 28. CDC CHECK - WTMEC2YR
# ============================================================

#
# Genuine decimal MEC examination weight.
#
# 0 means participant was not MEC examined.
#
# DO NOT convert to Int64.
#

s = df["WTMEC2YR"]


mec_zero_count = int(
    s.eq(0)
    .sum()
)


mec_positive_count = int(
    s.gt(0)
    .sum()
)


mec_missing_count = int(
    s.isna()
    .sum()
)


print("\n--- WTMEC2YR ---")


print(
    "Zero weights:",
    mec_zero_count
)


print(
    "Positive weights:",
    mec_positive_count
)


print(
    "Missing:",
    mec_missing_count
)


if not (
    mec_zero_count == 362
    and
    mec_positive_count == 9813
    and
    mec_missing_count == 0
):

    raise ValueError(
        "WTMEC2YR differs from expected release."
    )


print(
    "PASS: WTMEC2YR has exactly "
    "362 valid zero MEC weights."
)


# ============================================================
# 29. WTMEC2YR / RIDSTATR CONSISTENCY
# ============================================================

#
# WTMEC2YR=0 should correspond exactly to:
#
#     RIDSTATR=1
#
# interview-only participants.
#

mec_zero = (
    df["WTMEC2YR"]
    .eq(0)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


interview_only = (
    df["RIDSTATR"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


mismatch_count = int(
    np.sum(
        mec_zero
        !=
        interview_only
    )
)


print("\n--- MEC WEIGHT ROUTING CHECK ---")


print(
    "WTMEC2YR=0 vs RIDSTATR=1 mismatches:",
    mismatch_count
)


if mismatch_count != 0:

    raise ValueError(
        "WTMEC2YR=0 does not exactly match "
        "RIDSTATR=1."
    )


print(
    "PASS: All 362 WTMEC2YR zeros correspond "
    "exactly to interview-only participants."
)


# ============================================================
# 30. RIDEXMON / RIDSTATR CONSISTENCY
# ============================================================

ridexmon_missing = (
    df["RIDEXMON"]
    .isna()
    .to_numpy(
        dtype=bool
    )
)


mismatch_count = int(
    np.sum(
        ridexmon_missing
        !=
        interview_only
    )
)


if mismatch_count != 0:

    raise ValueError(
        "RIDEXMON missing pattern does not "
        "match interview-only participants."
    )


print(
    "PASS: RIDEXMON missing pattern matches "
    "the 362 interview-only participants."
)


# ============================================================
# 31. WTINT2YR CHECK
# ============================================================

#
# Genuine decimal interview weight.
#
# Preserve exactly.
#

s = df["WTINT2YR"]


wtint_nonmissing = int(
    s.notna()
    .sum()
)


wtint_missing = int(
    s.isna()
    .sum()
)


print("\n--- WTINT2YR ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Nonmissing:",
    wtint_nonmissing
)


print(
    "Missing:",
    wtint_missing
)


if wtint_nonmissing != 10175:

    raise ValueError(
        "Unexpected WTINT2YR nonmissing count."
    )


if wtint_missing != 0:

    raise ValueError(
        "Unexpected WTINT2YR missing values."
    )


if not (
    s.gt(0)
    .all()
):

    raise ValueError(
        "WTINT2YR contains unexpected "
        "zero/negative weight."
    )


print(
    "PASS: WTINT2YR pattern validated."
)


# ============================================================
# 32. CDC CHECK - INDFMPIR
# ============================================================

#
# Family income-to-poverty ratio.
#
# Genuine decimal variable.
#
# Expected release:
#
#     0 <= value < 5 : 7,974
#     value = 5.00   : 1,416
#     Missing        :   785
#
# Attached XPT:
#
#     restored zero = 105
#
# CRITICAL:
#
# DO NOT convert INDFMPIR to Int64.
# DO NOT round INDFMPIR.
#

s = df["INDFMPIR"]


print("\n--- INDFMPIR ---")


#
# Count actual zeros.
#
zero_count = int(
    s.eq(0)
    .sum()
)


#
# FIXED:
#
# Sum the Boolean Series BEFORE converting to int.
#
range_0_to_under_5_count = int(
    (
        s.ge(0)
        &
        s.lt(5)
    )
    .sum()
)


five_count = int(
    s.eq(5)
    .sum()
)


missing_count = int(
    s.isna()
    .sum()
)


print(
    "Restored zeros:",
    zero_count
)


print(
    "Values 0 through <5:",
    range_0_to_under_5_count
)


print(
    "Values equal to 5.00:",
    five_count
)


print(
    "Missing:",
    missing_count
)


if range_0_to_under_5_count != 7974:

    raise ValueError(
        "INDFMPIR 0-4.99 count differs.\n"
        f"Expected: 7,974\n"
        f"Actual:   {range_0_to_under_5_count:,}"
    )


if five_count != 1416:

    raise ValueError(
        "INDFMPIR=5 count differs.\n"
        f"Expected: 1,416\n"
        f"Actual:   {five_count:,}"
    )


if zero_count != 105:

    raise ValueError(
        "INDFMPIR zero count differs.\n"
        f"Expected: 105\n"
        f"Actual:   {zero_count:,}"
    )


if missing_count != 785:

    raise ValueError(
        "INDFMPIR missing count differs.\n"
        f"Expected: 785\n"
        f"Actual:   {missing_count:,}"
    )


#
# Additional released-range protection.
#
# Every nonmissing value must be between
# 0 and 5 inclusive.
#

invalid_range_count = int(
    (
        s.notna()
        &
        (
            s.lt(0)
            |
            s.gt(5)
        )
    )
    .sum()
)


if invalid_range_count != 0:

    raise ValueError(
        f"INDFMPIR contains "
        f"{invalid_range_count:,} value(s) "
        "outside the expected 0-5 range."
    )


print(
    "PASS: INDFMPIR range, zero count, "
    "5.00 count and missing count validated."
)


print(
    "PASS: INDFMPIR remains a genuine "
    "decimal field and was not rounded."
)


# ============================================================
# 33. MASKED VARIANCE UNIT CHECK
# ============================================================

#
# SDMVPSU:
#     1-2
#
# SDMVSTRA:
#     104-118
#
# 15 strata in this cycle.
#

if not (
    df["SDMVPSU"]
    .dropna()
    .between(
        1,
        2
    )
    .all()
):

    raise ValueError(
        "Unexpected SDMVPSU value."
    )


if not (
    df["SDMVSTRA"]
    .dropna()
    .between(
        104,
        118
    )
    .all()
):

    raise ValueError(
        "Unexpected SDMVSTRA value."
    )


if int(
    df["SDMVSTRA"]
    .nunique()
) != 15:

    raise ValueError(
        "Expected exactly 15 masked variance strata."
    )


print(
    "PASS: SDMVPSU/SDMVSTRA "
    "release structure validated."
)


# ============================================================
# 34. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Raw pandas-decoded XPT contained no ordinary
# numeric zeros.
#
# Therefore every zero after correction should
# belong exactly to the nine validated variables.
#

final_zero_counts = {}


for col in numeric_cols:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated DEMO_H pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly matches "
    "the nine validated corrections."
)


# ============================================================
# 35. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY allowed numeric changes from the raw
# pandas-decoded source are:
#
#     19,889 validated TINY_VALUE -> 0 corrections.
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 47 variables match the expected "
    "corrected XPT exactly."
)


# ============================================================
# 36. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# This preserves natural decimal values in:
#
# WTINT2YR
# WTMEC2YR
# INDFMPIR
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")


print(
    csv_file
)


# ============================================================
# 37. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 38. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 47 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 39. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 40. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 41. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 42. .0 CHECK - WHOLE-NUMBER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT globally remove ".0".
#
# Genuine decimal fields:
#
# WTINT2YR
# WTMEC2YR
# INDFMPIR
#
# Only check ".0" in the 44 proven integer fields.
#

print("\n--- INTEGER COLUMN .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = (
        raw_csv_df[col]
    )


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 in "
    "the 44 whole-number fields."
)


# ============================================================
# 43. LEGITIMATE DECIMAL .0 VALUES
# ============================================================

#
# Informational only.
#
# ".0" is allowed in genuine decimal columns.
#
# Examples:
#
# WTMEC2YR = 0.0
# INDFMPIR = 5.0
#
# Do NOT treat those as fidelity errors.
#

print(
    "\n--- LEGITIMATE DECIMAL .0 TOKENS ---"
)


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    decimal_dot_zero_counts[col] = count


    print(
        f"{col}: {count:,}"
    )


print(
    "PASS: Decimal fields were not "
    "globally stripped."
)


# ============================================================
# 44. EXACT DECIMAL FIDELITY
# ============================================================

#
# Check the three real decimal fields separately.
#

print("\n--- EXACT DECIMAL FIDELITY ---")


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        raise ValueError(
            f"{col} decimal fidelity failed."
        )


    print(
        f"PASS: {col} preserved exactly."
    )


# ============================================================
# 45. FINAL CSV ZERO-PATTERN CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly "
    "matches validated corrections."
)


# ============================================================
# 46. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 47. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 73557
    and
    final_seqn.max() == 83731
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 48. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "DEMO_H CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "RIDAGEYR zero restorations:",
    EXPECTED_TINY_COUNTS["RIDAGEYR"]
)


print(
    "RIDAGEMN zero restorations:",
    EXPECTED_TINY_COUNTS["RIDAGEMN"]
)


print(
    "RIDEXAGM zero restorations:",
    EXPECTED_TINY_COUNTS["RIDEXAGM"]
)


print(
    "DMDEDUC3 zero restorations:",
    EXPECTED_TINY_COUNTS["DMDEDUC3"]
)


print(
    "DMDHHSZA zero restorations:",
    EXPECTED_TINY_COUNTS["DMDHHSZA"]
)


print(
    "DMDHHSZB zero restorations:",
    EXPECTED_TINY_COUNTS["DMDHHSZB"]
)


print(
    "DMDHHSZE zero restorations:",
    EXPECTED_TINY_COUNTS["DMDHHSZE"]
)


print(
    "WTMEC2YR zero restorations:",
    EXPECTED_TINY_COUNTS["WTMEC2YR"]
)


print(
    "INDFMPIR zero restorations:",
    EXPECTED_TINY_COUNTS["INDFMPIR"]
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0 in integer fields: "
    "0 expected"
)


print(
    "Decimal values altered: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 DEMO_H XPT -> CSV
Rows: 10,175
Columns: 47
PASS: Structure = 10,175 rows x 47 columns.

--- VARIABLE TYPE PLAN ---
Genuine decimal fields: ['WTINT2YR', 'WTMEC2YR', 'INDFMPIR']
Whole-number fields: 44

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 19,889

Tiny values by variable:
DMDHHSZE    7384
DMDHHSZA    6417
DMDHHSZB    4907
RIDAGEYR     405
WTMEC2YR     362
DMDEDUC3     277
INDFMPIR     105
RIDAGEMN      28
RIDEXAGM       4

PASS: Exact 19,889-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
RIDAGEYR: 405 restored to 0
RIDAGEMN: 28 restored to 0
RIDEXAGM: 4 restored to 0
DMDEDUC3: 277 restored to 0
DMDHHSZA: 6,417 restored to 0
DMDHHSZB: 4,907 restored to 0
DMDHHSZE: 7,384 restored to 0
WTMEC2YR: 362 restored to 0
INDFMPIR: 105 restored to 0

PASS: 19,889 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL CHECK ---
WTINT2YR: 10,175 